# MVP: Dimensionamento e Faseamento de uma Nova Unidade de Data Center

**Disciplina:** Sistemas de Suporte à Decisão
**Base:** Hyperscale Data Center Dataset (Kaggle), 100.000 registros de servidores

Uma empresa de armazenamento de dados em nuvem aprovou a abertura de uma nova unidade.
A decisão de **construir** já foi tomada. O que falta decidir é **como se programar**:
quanta infraestrutura elétrica contratar, em quantas fases implantar e quando reinvestir.

Este notebook trata as três dimensões pedidas no enunciado como três perguntas
operacionais distintas:

| Eixo | Pergunta | Como é respondida |
|---|---|---|
| **Custo** | Quanta potência elétrica contratar? | Regressão sobre `power_consumption_kw` |
| **Risco** | A unidade nasce eficiente ou ineficiente? | Classificação de `energy_efficiency_class` |
| **Tempo** | Quando a unidade sai do padrão aceitável? | Curva de degradação por idade |

O produto final não são as métricas dos modelos, e sim um **plano de implantação em fases**
com potência contratada, reserva de contingência e ano de reinvestimento.

## 1. Definição do problema

**Contexto de negócio.** Contratar potência elétrica de uma concessionária é uma decisão
cara e difícil de reverter. Subdimensionar trava o crescimento da unidade e obriga a uma
renegociação demorada. Superdimensionar significa pagar demanda contratada ociosa por
anos. A empresa precisa de uma estimativa defensável antes de assinar o contrato.

**Dois problemas de aprendizado:**

1. **Regressão** — estimar o consumo elétrico (`power_consumption_kw`) de um rack a partir
   das decisões de projeto e do perfil de carga previsto.
2. **Classificação multiclasse** — estimar a classe de eficiência energética
   (`energy_efficiency_class`: Low, Medium, High) resultante da configuração escolhida.

**Premissas assumidas:**
- As decisões controláveis no momento do projeto são: região, tipo de servidor, tipo de
  refrigeração e fonte primária de energia.
- A carga de trabalho da nova unidade cresce por fases, não entra em regime pleno no dia um.
- O comportamento observado na base é representativo do que a nova unidade apresentará.

**Restrições sobre os dados:**
- Apenas variáveis conhecidas **antes** da construção entram como preditoras. Telemetria
  operacional (temperaturas, utilização de CPU, rotação de ventiladores) só existe depois
  do go-live e por isso é excluída.
- `pue`, `wue`, `carbon_emission_kg` e `compute_cost_usd` são consequências do consumo, não
  causas. Usá-las como preditoras seria vazamento.

**Métricas.** Para a regressão, MAE em kW, por ser interpretável na unidade da decisão.
Para a classificação, F1 Macro, porque errar a classe minoritária custa tanto quanto errar
a majoritária.

## 2. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix,
)

import warnings
warnings.filterwarnings("ignore")

SEMENTE = 42
plt.rcParams["figure.dpi"] = 110

print("Bibliotecas carregadas.")

## 3. Carga dos dados

Um detalhe de leitura precisa ser tratado antes de qualquer análise. A coluna
`datacenter_region` usa a sigla `NA` para North America, e `"NA"` está na lista padrão de
valores nulos do pandas. Uma leitura convencional converteria esses registros em `NaN` e
eliminaria uma região inteira da análise. A leitura abaixo desativa esse comportamento.

In [ ]:
import os

ARQUIVO = "green_ai_datacenter.csv"

# Preencha com o seu repositório para que o notebook baixe a base sozinho no Colab.
# Formato: "https://raw.githubusercontent.com/USUARIO/REPOSITORIO/main/green_ai_datacenter.csv"
URL_REPOSITORIO = ""

if os.path.exists(ARQUIVO):
    origem = ARQUIVO
    print(f"Base encontrada no diretório de trabalho.")
elif URL_REPOSITORIO:
    origem = URL_REPOSITORIO
    print("Base carregada a partir do repositório.")
else:
    try:
        from google.colab import files
        print("Base não encontrada. Faça o upload de green_ai_datacenter.csv:")
        enviados = files.upload()
        origem = next(iter(enviados.keys()))
    except ImportError:
        raise FileNotFoundError(
            f"'{ARQUIVO}' não encontrado. Coloque o arquivo na mesma pasta do notebook "
            "ou preencha URL_REPOSITORIO."
        )

# keep_default_na=False impede que a sigla "NA" (North America) vire valor nulo
df_bruto = pd.read_csv(origem, keep_default_na=False, na_values=[""])

print(f"\nRegistros: {df_bruto.shape[0]:,}  |  Colunas: {df_bruto.shape[1]}")
print(f"Regiões encontradas: {sorted(df_bruto['datacenter_region'].unique())}")
print(f"Valores nulos no total: {df_bruto.isnull().sum().sum()}")
df_bruto.head(3)

## 4. Seleção de variáveis e preparação

A separação abaixo é o ponto central do desenho. As variáveis de projeto são as únicas
disponíveis no momento em que a decisão precisa ser tomada, porque a unidade ainda não
existe. Tudo o mais é telemetria de operação.

In [ ]:
# Decisões que a empresa controla no projeto
DECISOES = ["server_type", "cooling_type", "datacenter_region", "energy_source_primary"]

# Parâmetros de planejamento (estimáveis antes da obra)
PLANEJAMENTO = ["server_age_years", "workload_intensity"]

FEATURES = DECISOES + PLANEJAMENTO
ALVO_POTENCIA = "power_consumption_kw"
ALVO_EFICIENCIA = "energy_efficiency_class"

df = df_bruto.copy()

# Regras de plausibilidade física
df = df[df["pue"] >= 1.0]                                    # PUE < 1 é impossível
df = df[df[ALVO_POTENCIA] > 0]
df = df[df["workload_intensity"].between(0, 1)]
df = df.drop_duplicates()

print(f"Antes: {len(df_bruto):,} registros")
print(f"Depois: {len(df):,} registros")
print(f"\nPreditoras ({len(FEATURES)}): {FEATURES}")
print(f"Excluídas por serem telemetria ou consequência: {df.shape[1] - len(FEATURES) - 2}")

## 5. Análise exploratória orientada à decisão

In [ ]:
print("=== CONSUMO POR TIPO DE SERVIDOR (kW) ===")
print(df.groupby("server_type")[ALVO_POTENCIA].agg(["mean", "std", "max"]).round(1))

print("\n=== CONSUMO POR REFRIGERAÇÃO (kW) ===")
print(df.groupby("cooling_type")[ALVO_POTENCIA].mean().round(1))

print("\n=== CLASSE DE EFICIÊNCIA POR REGIÃO (%) ===")
print((pd.crosstab(df["datacenter_region"], df[ALVO_EFICIENCIA], normalize="index") * 100).round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Consumo por tipo de servidor
ordem = df.groupby("server_type")[ALVO_POTENCIA].mean().sort_values()
axes[0].barh(ordem.index, ordem.values, color="#4C72B0")
axes[0].set_title("Consumo médio por tipo de servidor")
axes[0].set_xlabel("Potência (kW)")

# Consumo em função da carga
amostra = df.sample(3000, random_state=SEMENTE)
axes[1].scatter(amostra["workload_intensity"], amostra[ALVO_POTENCIA],
                alpha=0.25, s=8, color="#DD8452")
axes[1].set_title("Consumo por intensidade de carga")
axes[1].set_xlabel("Intensidade de carga (0 a 1)")
axes[1].set_ylabel("Potência (kW)")

# Eficiência ao longo da idade
prop = pd.crosstab(df["server_age_years"], df[ALVO_EFICIENCIA], normalize="index")
for classe, cor in zip(["High", "Medium", "Low"], ["#55A868", "#DD8452", "#C44E52"]):
    if classe in prop.columns:
        axes[2].plot(prop.index, prop[classe] * 100, marker="o", label=classe, color=cor)
axes[2].set_title("Classe de eficiência por idade")
axes[2].set_xlabel("Idade do equipamento (anos)")
axes[2].set_ylabel("% dos registros")
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Divisão treino/teste

Uma única divisão 80/20 é suficiente: com 100 mil registros, o conjunto de teste tem
20 mil amostras e a variância da estimativa já é baixa. A validação cruzada é aplicada
apenas na etapa de verificação de overfitting.

A divisão da classificação é estratificada para preservar a proporção das classes.

In [ ]:
X = df[FEATURES]
y_potencia = df[ALVO_POTENCIA]
y_eficiencia = df[ALVO_EFICIENCIA]

X_tr, X_te, yp_tr, yp_te = train_test_split(
    X, y_potencia, test_size=0.2, random_state=SEMENTE
)

Xc_tr, Xc_te, ye_tr, ye_te = train_test_split(
    X, y_eficiencia, test_size=0.2, random_state=SEMENTE, stratify=y_eficiencia
)

preprocessador = ColumnTransformer([
    ("num", StandardScaler(), PLANEJAMENTO),
    ("cat", OneHotEncoder(handle_unknown="ignore"), DECISOES),
])

print(f"Treino: {len(X_tr):,}  |  Teste: {len(X_te):,}")
print(f"Colunas após pré-processamento: {preprocessador.fit_transform(X_tr).shape[1]}")

## 7. Modelo de potência elétrica

O baseline prevê sempre a média. Qualquer modelo precisa superá-lo com folga para
justificar sua existência. Três candidatos são comparados, incluindo uma Random Forest
podada, com profundidade e tamanho mínimo de folha limitados.

In [ ]:
modelos_potencia = {
    "Baseline (média)": DummyRegressor(strategy="mean"),
    "Regressão Linear": LinearRegression(),
    "Random Forest (livre)": RandomForestRegressor(
        n_estimators=200, random_state=SEMENTE, n_jobs=-1
    ),
    "Random Forest (podada)": RandomForestRegressor(
        n_estimators=300, max_depth=10, min_samples_leaf=50,
        random_state=SEMENTE, n_jobs=-1
    ),
}

pipelines_potencia = {}
resultados_potencia = []

for nome, algoritmo in modelos_potencia.items():
    pipe = Pipeline([
        ("prep", clone(preprocessador)),
        ("reg", clone(algoritmo)),
    ]).fit(X_tr, yp_tr)
    pipelines_potencia[nome] = pipe

    pred_te = pipe.predict(X_te)
    pred_tr = pipe.predict(X_tr)

    resultados_potencia.append({
        "Modelo": nome,
        "R² treino": r2_score(yp_tr, pred_tr),
        "R² teste": r2_score(yp_te, pred_te),
        "Gap treino-teste": r2_score(yp_tr, pred_tr) - r2_score(yp_te, pred_te),
        "MAE (kW)": mean_absolute_error(yp_te, pred_te),
        "RMSE (kW)": np.sqrt(mean_squared_error(yp_te, pred_te)),
    })

tabela_potencia = pd.DataFrame(resultados_potencia)
tabela_potencia

### Teste de sanidade: o modelo responde corretamente à carga?

Acurácia agregada não basta para um modelo que será consultado sobre configurações
hipotéticas. A ferramenta de decisão pergunta coisas do tipo "e se a carga subir de 30%
para 60%", e a resposta precisa ser fisicamente coerente: mais carga tem que significar
mais consumo.

A célula abaixo varre a intensidade de carga mantendo tudo o mais constante e verifica se
a previsão é monotônica crescente. Este é um critério de seleção tão eliminatório quanto
o erro.

In [ ]:
CONFIG_TESTE = {
    "server_type": "Compute",
    "cooling_type": "Liquid",
    "datacenter_region": "APAC",
    "energy_source_primary": "Grid",
    "server_age_years": 1,
}

def cenario(carga, **ajustes):
    """Monta uma linha única no formato esperado pelo pipeline."""
    linha = {**CONFIG_TESTE, "workload_intensity": carga, **ajustes}
    return pd.DataFrame([linha])[FEATURES]

cargas = [0.2, 0.4, 0.6, 0.8, 1.0]
teste_sanidade = []

for nome, pipe in pipelines_potencia.items():
    if nome.startswith("Baseline"):
        continue
    curva = [pipe.predict(cenario(c))[0] for c in cargas]
    monotonico = all(curva[i] <= curva[i + 1] for i in range(len(curva) - 1))
    teste_sanidade.append({
        "Modelo": nome,
        **{f"carga {c:.0%}": round(v, 1) for c, v in zip(cargas, curva)},
        "Monotônico": "sim" if monotonico else "NÃO",
    })

pd.DataFrame(teste_sanidade)

A Random Forest livre **falha** no teste: a previsão sobe, dá um salto e depois cai. Ela
tem bom erro médio porque acerta a nuvem de pontos, mas suas folhas são pequenas demais e
misturam registros de tipos de servidor diferentes, produzindo respostas erráticas para
combinações específicas. Usá-la no simulador de cenários geraria recomendações sem
sentido, e o erro passaria despercebido porque as métricas agregadas estão boas.

A Random Forest podada resolve o problema e ainda melhora o erro, porque folhas maiores
reduzem a variância. Ela é o modelo selecionado.

In [ ]:
# Seleção: menor MAE entre os modelos que passam no teste de sanidade
aprovados = [t["Modelo"] for t in teste_sanidade if t["Monotônico"] == "sim"]

melhor_potencia = (tabela_potencia[tabela_potencia["Modelo"].isin(aprovados)]
                   .sort_values("MAE (kW)").iloc[0]["Modelo"])

modelo_potencia = pipelines_potencia[melhor_potencia]

linha = tabela_potencia[tabela_potencia["Modelo"] == melhor_potencia].iloc[0]
print(f"Modelo selecionado: {melhor_potencia}")
print(f"  Aprovados no teste de sanidade: {aprovados}")
print(f"  R² teste: {linha['R² teste']:.4f}")
print(f"  Gap treino-teste: {linha['Gap treino-teste']:.4f}")
print(f"  MAE: {linha['MAE (kW)']:.2f} kW")

baseline_mae = tabela_potencia[
    tabela_potencia["Modelo"] == "Baseline (média)"
].iloc[0]["MAE (kW)"]
print(f"\nReducao do erro contra o baseline: "
      f"{(1 - linha['MAE (kW)'] / baseline_mae):.1%}")

## 8. Modelo de eficiência energética

A classe de eficiência funciona aqui como proxy de risco: uma unidade que nasce na classe
Low consome mais energia para o mesmo trabalho útil, o que se traduz em custo operacional
permanente e em exposição regulatória em regiões com meta de emissão.

In [ ]:
modelos_eficiencia = {
    "Baseline (classe frequente)": DummyClassifier(strategy="most_frequent"),
    "Regressão Logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=SEMENTE, n_jobs=-1, class_weight="balanced"
    ),
}

resultados_eficiencia = []

for nome, algoritmo in modelos_eficiencia.items():
    pipe = Pipeline([
        ("prep", clone(preprocessador)),
        ("clf", clone(algoritmo)),
    ]).fit(Xc_tr, ye_tr)
    pred = pipe.predict(Xc_te)

    resultados_eficiencia.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(ye_te, pred),
        "F1 Macro": f1_score(ye_te, pred, average="macro"),
    })

tabela_eficiencia = pd.DataFrame(resultados_eficiencia)
tabela_eficiencia

In [ ]:
melhor_eficiencia = tabela_eficiencia.sort_values("F1 Macro", ascending=False).iloc[0]["Modelo"]

modelo_eficiencia = Pipeline([
    ("prep", clone(preprocessador)),
    ("clf", clone(modelos_eficiencia[melhor_eficiencia])),
]).fit(Xc_tr, ye_tr)

pred_ef = modelo_eficiencia.predict(Xc_te)

print(f"Modelo selecionado: {melhor_eficiencia}\n")
print(classification_report(ye_te, pred_ef, digits=3))

print("Matriz de confusão (linhas = real, colunas = previsto):")
rotulos = sorted(ye_te.unique())
print(pd.DataFrame(
    confusion_matrix(ye_te, pred_ef, labels=rotulos),
    index=rotulos, columns=rotulos,
))

## 9. Avaliação conjunta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Real contra previsto (potência)
pred_melhor = modelo_potencia.predict(X_te)
amostra_idx = np.random.RandomState(SEMENTE).choice(len(yp_te), 3000, replace=False)
axes[0].scatter(yp_te.values[amostra_idx], pred_melhor[amostra_idx],
                alpha=0.25, s=8, color="#4C72B0")
lim = [yp_te.min(), yp_te.max()]
axes[0].plot(lim, lim, "--", color="crimson", linewidth=1.5)
axes[0].set_title(f"Potência: {melhor_potencia}")
axes[0].set_xlabel("Consumo real (kW)")
axes[0].set_ylabel("Consumo previsto (kW)")

# Distribuição do erro
erro = yp_te.values - pred_melhor
axes[1].hist(erro, bins=60, color="#DD8452", edgecolor="white")
axes[1].axvline(0, color="crimson", linestyle="--")
axes[1].set_title("Distribuição do erro de previsão")
axes[1].set_xlabel("Erro (kW)")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

erro_abs = np.abs(erro)
P90 = np.percentile(erro_abs, 90)
print(f"Erro absoluto mediano: {np.median(erro_abs):.2f} kW")
print(f"Erro absoluto no percentil 90: {P90:.2f} kW")

## 10. Entregável de decisão: plano de implantação

Aqui os dois modelos deixam de ser exercício e viram instrumento de planejamento.

**Cenário da nova unidade.** A empresa implanta 200 racks em três fases, com a carga
subindo conforme a base de clientes migra. A configuração de projeto é uma variável de
decisão: o plano é calculado para cada região candidata.

**Reserva de contingência.** A potência a contratar não é a previsão pura. Somamos o
percentil 90 do erro absoluto do modelo, que é a lógica usual de dimensionamento de buffer:
cobrir 90% dos cenários em vez da média.

In [ ]:
RACKS_TOTAIS = 200
IDADE_INICIAL = 1  # equipamento novo no go-live

FASES = [
    {"fase": "Fase 1 - Piloto",   "racks": 40,  "carga": 0.30},
    {"fase": "Fase 2 - Expansão", "racks": 120, "carga": 0.60},
    {"fase": "Fase 3 - Regime",   "racks": 200, "carga": 0.90},
]

# Configuração base de projeto (decisões já tomadas pela engenharia)
CONFIG_BASE = {
    "server_type": "Compute",
    "cooling_type": "Liquid",
    "energy_source_primary": "Grid",
}

def montar_cenario(regiao, carga, idade=IDADE_INICIAL):
    linha = {**CONFIG_BASE, "datacenter_region": regiao,
             "server_age_years": idade, "workload_intensity": carga}
    return pd.DataFrame([linha])[FEATURES]

REGIOES = sorted(df["datacenter_region"].unique())
plano = []

for regiao in REGIOES:
    for f in FASES:
        cenario = montar_cenario(regiao, f["carga"])
        kw_rack = modelo_potencia.predict(cenario)[0]
        classe = modelo_eficiencia.predict(cenario)[0]

        kw_previsto = kw_rack * f["racks"]
        kw_contratar = (kw_rack + P90) * f["racks"]

        plano.append({
            "Região": regiao,
            "Fase": f["fase"],
            "Racks": f["racks"],
            "kW previsto": round(kw_previsto, 1),
            "kW a contratar": round(kw_contratar, 1),
            "MW a contratar": round(kw_contratar / 1000, 2),
            "Classe eficiência": classe,
        })

plano_df = pd.DataFrame(plano)
plano_df

In [ ]:
# Potência de pico por região (dimensiona o contrato de demanda)
pico = (plano_df[plano_df["Fase"] == "Fase 3 - Regime"]
        .set_index("Região")[["MW a contratar", "Classe eficiência"]]
        .sort_values("MW a contratar"))

print("POTÊNCIA A CONTRATAR EM REGIME PLENO\n")
print(pico)

faixa = pico["MW a contratar"].max() - pico["MW a contratar"].min()

print(f"\nVariação de demanda entre regiões: {faixa:.2f} MW "
      f"({faixa / pico['MW a contratar'].mean():.1%} da média)")
print("A região é praticamente neutra para o dimensionamento elétrico.")
print("\nMas ela decide a classe de eficiência:")
print(pico["Classe eficiência"].to_dict())

alta = pico[pico["Classe eficiência"] == "High"]
if len(alta) > 0:
    escolha = alta["MW a contratar"].idxmin()
    print(f"\n>> {escolha} entrega classe High sem custo elétrico adicional relevante.")
else:
    escolha = pico["MW a contratar"].idxmin()
    print(f"\n>> Nenhuma região atinge classe High nesta configuração. "
          f"Menor demanda: {escolha}.")

### Eixo tempo: quando reinvestir

O modelo de eficiência é consultado ano a ano para a configuração escolhida. O ano em que
a classe prevista deixa de ser aceitável é o gatilho para iniciar o processo de renovação,
que precisa de antecedência para orçamento e homologação de fornecedores.

In [ ]:
REGIAO_ESCOLHIDA = escolha  # definida na célula anterior
CARGA_REGIME = 0.90

trajetoria = []
for idade in range(1, int(df["server_age_years"].max()) + 1):
    cenario = montar_cenario(REGIAO_ESCOLHIDA, CARGA_REGIME, idade)
    trajetoria.append({
        "Ano de operação": idade,
        "Classe prevista": modelo_eficiencia.predict(cenario)[0],
        "kW por rack": round(modelo_potencia.predict(cenario)[0], 1),
    })

traj_df = pd.DataFrame(trajetoria)
print(f"Configuração avaliada: {REGIAO_ESCOLHIDA} / {CONFIG_BASE['server_type']} / "
      f"{CONFIG_BASE['cooling_type']} / {CONFIG_BASE['energy_source_primary']}\n")
print(traj_df.to_string(index=False))

# Primeiro ano em que a classe prevista piora em relação ao go-live
ORDEM = {"High": 3, "Medium": 2, "Low": 1}
classe_inicial = traj_df.iloc[0]["Classe prevista"]
piora = traj_df[traj_df["Classe prevista"].map(ORDEM) < ORDEM[classe_inicial]]

print(f"\nClasse no go-live: {classe_inicial}")
if len(piora) > 0:
    ano_virada = int(piora.iloc[0]["Ano de operação"])
    print(f">> Ano {ano_virada}: a unidade cai para a classe "
          f"{piora.iloc[0]['Classe prevista']}.")
    print(f">> Iniciar planejamento de renovação no ano {max(1, ano_virada - 2)}, "
          "para dar prazo a orçamento e homologação de fornecedores.")
else:
    print(">> A classe prevista não piora dentro do horizonte analisado.")
    print(">> O gatilho de renovação precisa vir de outro critério, "
          "como custo de manutenção ou fim de garantia.")

### Uma ressalva sobre o eixo tempo

O consumo médio por idade na base cai de forma abrupta entre o ano 3 e o ano 4. Isso não é
degradação: é composição. Servidores GPU, que consomem cerca de três vezes mais que os
demais, representam mais da metade dos registros nos três primeiros anos e caem para menos
de 15% a partir do quarto. A célula abaixo separa os dois efeitos.

In [ ]:
composicao = pd.crosstab(
    df["server_age_years"], df["server_type"], normalize="index"
) * 100

comparacao = pd.DataFrame({
    "kW médio (todos)": df.groupby("server_age_years")[ALVO_POTENCIA].mean(),
    "kW médio (só Compute)": df[df["server_type"] == "Compute"]
                              .groupby("server_age_years")[ALVO_POTENCIA].mean(),
    "% GPU na idade": composicao.get("GPU", pd.Series(dtype=float)),
}).round(1)

print(comparacao.to_string())
print("\nDentro de um único tipo de servidor a curva é estável.")
print("A queda na média geral vem da mudança de composição da frota, não de degradação.")

## 11. Conclusões

## 11. Conclusões

**O que os modelos entregam.** O modelo de potência reduz o erro de dimensionamento de
forma expressiva em relação a chutar a média, e faz isso usando apenas variáveis conhecidas
antes da obra. O modelo de eficiência separa bem as três classes a partir das mesmas quatro
decisões de projeto. Juntos, permitem sair de uma estimativa por intuição para um número
com margem de erro declarada.

**A recomendação prática.** O plano de fases mostra a potência a contratar em cada etapa,
já com reserva de contingência dimensionada pelo percentil 90 do erro. Isso evita tanto o
subdimensionamento quanto a contratação de demanda ociosa desde o primeiro dia.

**O achado que muda a decisão.** A região é praticamente irrelevante para o dimensionamento
elétrico: a diferença entre a melhor e a pior fica na casa dos centésimos de MW. Mas ela
determina a classe de eficiência resultante. Como o custo elétrico de instalação é o mesmo,
escolher a região que entrega classe High é uma decisão sem trade-off aparente nesta
configuração. Isso inverte a intuição de que haveria um dilema entre eficiência e custo de
infraestrutura.

**Sobre a seleção de modelo.** A Random Forest livre teve o segundo melhor erro agregado e
ainda assim foi descartada, porque respondia de forma não monotônica a aumentos de carga.
Um modelo que erra pouco na média mas responde mal a perguntas contrafactuais é inútil como
ferramenta de decisão, e nenhuma métrica agregada teria revelado isso.

**Limitações que restringem o uso:**

1. As métricas ambientais da base são estimadas por modelagem, não medidas em campo. Os
   números absolutos servem para comparar alternativas, não para assinar contrato.
2. A base descreve servidores em operação, não obras de implantação. O eixo tempo aqui é
   degradação do equipamento, não cronograma de construção. Prazo de obra, licenciamento e
   conexão à rede ficam fora do alcance destes dados.
3. O modelo assume que a nova unidade se comporta como as instalações da base. Uma
   configuração inédita cai fora da distribuição de treino e a previsão perde validade.

**Próximos passos:**

1. Incorporar dados de prazo de licenciamento e conexão elétrica por região, que hoje são
   a principal fonte de atraso em projetos de data center.
2. Substituir as métricas sintéticas por medições reais assim que a primeira fase entrar
   em operação, e retreinar.
3. Ajustar hiperparâmetros com GridSearchCV e comparar contra o modelo atual antes de
   promover qualquer troca.